# 0. Import modules

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics

from scipy.interpolate import interp1d

import datetime, os, json, random, shap, itertools, joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Data preprocessing

In [ ]:
capacity = pd.read_csv('./LMPO_discharge capacity_cycle.csv', index_col=0)

with open('./Formation_voltage.json', 'r', encoding='utf-8') as f:
    x_volt = json.load(f)

with open('./Formation_current.json', 'r', encoding='utf-8') as f:
    x_curr = json.load(f)
    
with open('./50th cycle_discharge voltage.json', 'r', encoding='utf-8') as f:
    y_volt = json.load(f)

In [ ]:
def resize_sequences(sequence, target_length):    
    current_len = sequence.shape[0]
    
    old_indices = np.linspace(0, 1, current_len)
    new_indices = np.linspace(0, 1, target_length)
    
    new_sequence = np.zeros((target_length, 1))
    for i in range(1):
        f = interp1d(old_indices, sequence[:], kind='linear')
        new_sequence[:, i] = f(new_indices)
        
    return np.array(new_sequence)

In [ ]:
resize_x_volt, resize_x_curr = {}, {}
len_x = 15

for i in x_volt.keys():
    temp = []
    for j in range(8):
        temp += list(resize_sequences(np.array(x_volt[i][j]), target_length=len_x).reshape([1, -1])[0])
    resize_x_volt[i] = np.array(temp)

    t = []
    for j in range(8):
        t += list(resize_sequences(np.array(x_curr[i][j]), target_length=len_x).reshape([1, -1])[0])
    resize_x_curr[i] = np.array([_*1000 for _ in t])

In [ ]:
rescaled_y_volt = {}
len_y = 20

for i in y_volt.keys():
    rescaled_y_volt[i] = resize_sequences(np.array(y_volt[i]), target_length=len_y+1).reshape([1, -1])[0][:-1]

In [ ]:
curr_scaler = joblib.load('./curr_scaler.pkl')
volt_scaler = joblib.load('./volt_scaler.pkl')

rescaled_x_volt, rescaled_x_curr = {}, {}
for i in resize_x_volt.keys():
    rescaled_x_volt[i] = volt_scaler.transform(resize_x_volt[i].reshape(-1, 1)).reshape(1, -1)[0]
    rescaled_x_curr[i] = curr_scaler.transform(resize_x_curr[i].reshape(-1, 1)).reshape(1, -1)[0]

# 2. Cycle life prediciton model

In [ ]:
BATCH_SIZE = 32

AUTOTUNE = tf.data.AUTOTUNE
def preprocess_fn(features, labels_tuple):
    features = tf.cast(features, tf.float32)
    return features, labels_tuple

train_key = sorted(random.sample(list(rescaled_x_volt.keys()), int(len(list(rescaled_x_volt.keys()))*0.85)))
test_key = sorted([_ for _ in list(rescaled_x_volt.keys()) if _ not in train_key])

X_train, X_test = [], []
y_train, y_test = [], []
for X_temp, key in zip([X_train, X_test], [train_key, test_key]):
    for i in key:
        temp_1 = []
        for j in range(len_x*8):
            temp_2 = np.array([rescaled_x_volt[i][j], rescaled_x_curr[i][j]])
            temp_1.append(temp_2)
        X_temp.append(np.array(temp_1))

y_volt_train = np.array(list(np.array(rescaled_y_volt[i]) for i in train_key))
y_volt_test = np.array(list(np.array(rescaled_y_volt[i]) for i in test_key))

y_capa_train, y_capa_test = [], []
for i in [1, 50]:
    y_capa_train.append(np.array(capacity.loc[train_key, f'{i}cyc']).reshape(-1, 1))
    y_capa_test.append(np.array(capacity.loc[test_key, f'{i}cyc']).reshape(-1, 1))

X_train, X_test = np.array(X_train), np.array(X_test)
X_all = np.concatenate([X_train, X_test])

train_dataset_raw = tf.data.Dataset.from_tensor_slices((X_train, (y_volt_train, tuple(y_capa_train))))
test_dataset_raw = tf.data.Dataset.from_tensor_slices((X_test, (y_volt_test, tuple(y_capa_test))))

train_dataset = (train_dataset_raw.shuffle(buffer_size=len(X_train)).map(preprocess_fn, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE))
train_dataset_plot = (train_dataset_raw.map(preprocess_fn, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE))
test_dataset = (test_dataset_raw.map(preprocess_fn, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE))

In [ ]:
def soft_clip_layer(x, min_val=1.5, max_val=4.8):
    activated = tf.sigmoid(x)
    return activated * (max_val - min_val) + min_val

input_layer = layers.Input(shape=(len_x*8, 2))

x = layers.Conv1D(filters=8, kernel_size=5, strides=2, activation='relu')(input_layer)
x = layers.MaxPooling1D(pool_size=3, strides=1)(x)
x = layers.Flatten()(x)

shared_layer_1_1 = layers.Dense(2**4, activation='leaky_relu', name='dense1')(x)
shared_layer_1_2 = layers.Dense(2**6, activation='leaky_relu', name='dense2')(shared_layer_1_1)
shared_layer_1_3 = layers.Dense(2**4, activation='leaky_relu', name='dense3')(shared_layer_1_2)
shared_layer_1_4 = layers.Dense(2**7, activation='leaky_relu', name='dense4')(shared_layer_1_3)
shared_layer_1_5 = layers.Dense(2**4, activation='leaky_relu', name='dense5')(shared_layer_1_4)

output_volt = layers.Dense(len_y, activation='linear', name='output_volt')(shared_layer_1_5)
clip_volt = layers.Lambda(lambda x: soft_clip_layer(x), name='clip_volt')(output_volt)

output_capa1 = layers.Dense(1, activation='linear', name='output_capa1')(shared_layer_1_4)
output_capa50 = layers.Dense(1, activation='linear', name='output_capa50')(shared_layer_1_4)

model = models.Model(inputs=input_layer, outputs=[clip_volt, (output_capa1, output_capa50)])
model.compile(optimizer='adam', loss={'clip_volt':'huber', 'output_capa1':'huber', 'output_capa50':'huber'}, 
              loss_weights={'clip_volt':50, 'output_capa1':1, 'output_capa50':1})

early_stopping = EarlyStopping(monitor='loss', patience=300, restore_best_weights=True)

history = model.fit(train_dataset, epochs=200000, batch_size=BATCH_SIZE, callbacks=[early_stopping], verbose=False)

In [ ]:
result_train = model.predict(train_dataset_plot)
result_test = model.predict(test_dataset)

for d, k in enumerate([1, 50]):
    plt.figure(figsize=(5,5))
    plt.plot([min(capacity[f'{k}cyc']), max(capacity[f'{k}cyc'])], [min(capacity[f'{k}cyc']), max(capacity[f'{k}cyc'])], 'k--')
    for j, result, name in zip([train_key, test_key], [result_train, result_test], ['train', 'test']):
        true, predict = [], []
        for c, i in enumerate(j):
            true.append(capacity.loc[i, f'{k}cyc'])
            predict.append(result[1][d][c])

        plt.scatter(true, predict, alpha=0.7)
        print('({}) R2 : {}, MAE : {}, RMSE : {}'.format(name, round(metrics.r2_score(true, predict), 2), round(metrics.mean_absolute_error(true, predict), 2), round(metrics.mean_squared_error(true, predict, squared=False), 2)))
            
    plt.title(f'{k} cycle', fontsize=16)
    plt.xlabel('True capacity [mAh/g]', fontsize=15)
    plt.ylabel('Predicted capacity [mAh/g]', fontsize=15)
    plt.show()

# 3. Model analysis

## 0) Load trained model

In [ ]:
model = tf.keras.models.load_model('./Final_model.keras', safe_mode=False)

In [ ]:
layer_name = [_['name'] for _ in model.get_config()['layers']]
input_name = [_ for _ in layer_name if 'input' in _][0]
conv_name = [_ for _ in layer_name if 'conv1d' in _][0]
maxpooling_name = [_ for _ in layer_name if 'max_pooling1d' in _][0]
flatten_name = [_ for _ in layer_name if 'flatten' in _][0]

## 1) Grad-cam

In [ ]:
def get_flat_multi_head_gradcam(model, sample_data, last_conv_layer_name):
    all_heatmaps = []
    input_tensor = tf.convert_to_tensor(np.expand_dims(sample_data, axis=0), dtype=tf.float32)
    grad_model = tf.keras.models.Model(
        inputs=[model.inputs], outputs=[model.get_layer(last_conv_layer_name).output] + model.outputs
    )
    
    num_total_outputs = len(model.outputs) 
    for i in range(1, num_total_outputs):
        with tf.GradientTape() as tape:
            results = grad_model(input_tensor)
            conv_output = results[0]
            target_output = results[i + 1]
            class_channel = target_output[:, 0]

        grads = tape.gradient(class_channel, conv_output)
        
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1))
        heatmap = tf.reduce_sum(conv_output[0] * pooled_grads, axis=-1)

        max_abs_val = tf.reduce_max(tf.abs(heatmap))

        if max_abs_val > 1e-10:
            heatmap = heatmap / max_abs_val
        else:
            heatmap = heatmap

        all_heatmaps.append(heatmap.numpy())

    return np.array(all_heatmaps)

In [ ]:
def get_flat_multi_head_gradcam_volt(model, sample_data, last_conv_layer_name, step):
    input_tensor = tf.convert_to_tensor(np.expand_dims(sample_data, axis=0), dtype=tf.float32)
    grad_model = tf.keras.models.Model(
        inputs=[model.inputs], outputs=[model.get_layer(last_conv_layer_name).output] + model.outputs
    )

    with tf.GradientTape() as tape:
        results = grad_model(input_tensor)
        conv_output = results[0]
        A_output = results[1]    
        class_channel = A_output[0, step]

    grads = tape.gradient(class_channel, conv_output)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1))
    heatmap = tf.reduce_sum(conv_output[0] * pooled_grads, axis=-1)

    max_abs_val = tf.reduce_max(tf.abs(heatmap))

    if max_abs_val > 1e-10:
        heatmap = heatmap / max_abs_val
    else:
        heatmap = heatmap

    return heatmap.numpy()

In [ ]:
X = []
for i in list(rescaled_x_volt.keys()):
    temp_1 = []
    for j in range(len_x*8):
        temp_2 = np.array([rescaled_x_volt[i][j], rescaled_x_curr[i][j]])
        temp_1.append(temp_2)
    X.append(np.array(temp_1))

bo = 0.85

c1, c50, v = [], [], []
for j, k in enumerate(rescaled_x_volt.keys()):
    sample_data = X[j]

    all_heatmaps_1 = get_flat_multi_head_gradcam(model, sample_data, conv_name)[0]
    all_heatmaps_50 = get_flat_multi_head_gradcam(model, sample_data, conv_name)[1]
    all_heatmaps_vol = get_flat_multi_head_gradcam_volt(model, sample_data, conv_name, 4)

    for time in range(len(all_heatmaps_1)):
        if abs(all_heatmaps_1[time]) > bo:
            c1.append(rescaled_x_volt[k][time*2])
        
        if abs(all_heatmaps_50[time]) > bo:
            c50.append(rescaled_x_volt[k][time*2])

        if abs(all_heatmaps_vol[time]) > bo:
            v.append(rescaled_x_volt[k][time*2])

plt.hist([_*(4.8-1.5)+1.5 for _ in c1], bins=30, alpha=0.5, label='1cyc')
plt.hist([_*(4.8-1.5)+1.5 for _ in c50], bins=30, alpha=0.5, label='50cyc')
plt.hist([_*(4.8-1.5)+1.5 for _ in v], bins=50, alpha=0.5, label='4th volt')
plt.legend()
plt.show()

In [ ]:
for c, i in enumerate(['1cyc', '50cyc', '50th voltage (4th)'])
    plt.figure(figsize=(5,4))
    ch, dch = [], []
    for j, k in enumerate(rescaled_x_volt.keys()):
        sample_data = X[j]
        if c == 2:
            all_heatmaps = get_flat_multi_head_gradcam_volt(model, sample_data, conv_name, 4)
        else:
            all_heatmaps = get_flat_multi_head_gradcam(model, sample_data, conv_name)[c]

        for time in range(len(all_heatmaps)):
            if abs(all_heatmaps[time]) > bo:
                if rescaled_x_curr[k][time*2] > 0:
                    ch.append(rescaled_x_volt[k][time*2])
                elif rescaled_x_curr[k][time*2] < 0:
                    dch.append(rescaled_x_volt[k][time*2])        

    plt.title(i, fontsize=13)
    plt.xlabel('Voltage', fontsize=13)
    plt.ylabel('Count', fontsize=13)
    plt.hist([_*(4.8-1.5)+1.5 for _ in ch], bins=40, alpha=0.5, range=[1.5, 4.8], label='charge')
    plt.hist([_*(4.8-1.5)+1.5 for _ in dch], bins=40, alpha=0.5, range=[1.5, 4.8], label='discharge')
    plt.legend()
    plt.show()

## 2) SHAP

In [ ]:
flatten_layer_shape = model.get_layer(flatten_name).output.shape[1]

input_layer_shap = tf.keras.Input(shape=(flatten_layer_shape,))

shared_layer_1_shap = model.get_layer('dense1')(input_layer_shap)
shared_layer_2_shap = model.get_layer('dense2')(shared_layer_1_shap)
shared_layer_3_shap = model.get_layer('dense3')(shared_layer_2_shap)
shared_layer_4_shap = model.get_layer('dense4')(shared_layer_3_shap)
shared_layer_5_shap = model.get_layer('dense5')(shared_layer_4_shap)

output_volt_shap = model.get_layer('output_volt')(shared_layer_5_shap)
clip_volt_shap = model.get_layer('clip_volt')(output_volt_shap)

output_capa1_shap = model.get_layer('output_capa1')(shared_layer_4_shap)
output_capa50_shap = model.get_layer('output_capa50')(shared_layer_4_shap)

model_shap = models.Model(inputs=input_layer_shap, outputs=[clip_volt_shap, (output_capa1_shap, output_capa50_shap)])
#################
input_layer_flat = layers.Input(shape=(len_x*8, 2))

x_flat = model.get_layer(conv_name)(input_layer_flat)
x_flat = model.get_layer(maxpooling_name)(x_flat)
x_flat = model.get_layer(flatten_name)(x_flat)

flatten_extractor = tf.keras.models.Model(inputs=input_layer_flat, outputs=x_flat)
flatten_data = flatten_extractor.predict(X_all)

In [ ]:
feature_name = []
for j in range(58):
    for i in range(8):
        feature_name.append(f'Filter {i}_{j} idx')
            
for c, name in enumerate(['1cyc discharge capacity', '50cyc discharge capacity', '50cyc discharge voltage (80% SoC)']):
    if c == 2:
        explainer = shap.GradientExplainer(model=(model_shap.inputs, model_shap.outputs[2]), data=flatten_data)
    else:
        explainer = shap.GradientExplainer(model=(model_shap.inputs, model_shap.outputs[c]), data=flatten_data)
        
    shap_values = explainer.shap_values(flatten_data)

    visual_shap = []
    for i in shap_values:
        visual_shap.append(i.flatten()) 
    visual_shap = np.array(visual_shap)



    shap.summary_plot(visual_shap, flatten_data, feature_names=feature_name, max_display=5, show=False)
    plt.title(name)
    plt.show()

# 4. Formation data prediction model

In [ ]:
col = ['idx', 'current', 'ch1', 'ch2', 'dch1', 'dch2']

t, temp = [], []
for i in list(resize_x_curr.keys()):
    cond = [int(_) for _ in i.split('_') if _.isnumeric()][:4]
    if cond not in temp:
        t.append(i)
        temp.append(cond)

train_key = sorted(random.sample(t, int(len(t)*0.96)))
test_key = [_ for _ in t if _ not in train_key]

df_train, df_test = pd.DataFrame(), pd.DataFrame()
for temp_df, key in zip([df_train, df_test], [train_key, test_key]):
    for i in key:
        cond = [int(_) for _ in i.split('_') if _.isnumeric()]
        for j in range(120):
            c = len(temp_df)
            for k, l in enumerate(col[2:])
                temp_df.loc[c, l] = round(cond[k], 3)
            temp_df.loc[c, 'idx'] = j
            temp_df.loc[c, 'current'] = round(resize_x_curr[i][j], 3)
            temp_df.loc[c, 'voltage'] = resize_x_volt[i][j]

X_train, X_test = df_train[col], df_test[col]
y_train, y_test = df_train.loc[:, 'voltage'], df_test.loc[:, 'voltage']

linear_model = LinearRegression().fit(X_train, y_train)
trend_train = linear_model.predict(X_train)

rf = RandomForestRegressor(n_estimators=985, max_depth=17, min_samples_split=2, min_samples_leaf=1, max_features=0.72)
residuals = y_train - trend_train
rf.fit(X_train, residuals)

def predict_hybrid(linear_model, rf, X_new):
    trend_pred = linear_model.predict(X_new)
    residual_pred = rf.predict(X_new)
    return trend_pred + residual_pred

result_train = predict_hybrid(linear_model, rf, X_train)
test_pred = predict_hybrid(linear_model, rf, X_test)

print(f'R2 : {round(metrics.r2_score(y_test, test_pred), 3)}, MAE : {round(metrics.mean_absolute_error(y_test, test_pred), 3)} V, RMSE : {round(metrics.mean_squared_error(y_test, test_pred, squared=False), 3)} V')

In [ ]:
curr_list = [20, 30, 50, 80, 100, 150, 200, 300, 500]
names = '\n'.join(t)

rows = [c for c in itertools.product(curr_list, repeat=4)
        if f'_{c[0]}_{c[1]}_{c[2]}_{c[3]}_' not in names]

all_cond = pd.DataFrame(rows, columns=['ch1', 'ch2', 'dch1', 'dch2']).reindex(columns=col[2:])

In [ ]:
cols = [('ch1', 1), ('ch2', 1), ('dch1', -1), ('dch2', -1)]

all_formation = {}
for i in range(len(all_formation), len(all_cond)):
    temp = pd.DataFrame()
    for j in range(120):
        temp.loc[j, 'idx'] = j
        
        for k in col[2:]:
            temp.loc[j, k] = round(all_cond.loc[i, k], 3)
            
        name, sign = cols[(j // 15) % 4]
        temp.loc[j, 'current'] = round(sign * all_cond.loc[i, name], 3)

        to_predict = np.array(temp.loc[j, col]).reshape(1, -1)
        temp.loc[j, 'voltage'] = float(predict_hybrid(rf, linear_model, to_predict))

    name = '{}_{}_{}_{}'.format(all_cond.loc[i, 'ch1'], all_cond.loc[i, 'ch2'], all_cond.loc[i, 'dch1'], all_cond.loc[i, 'dch2'])
    temp_dict = {'current':list(temp['current']), 'voltage':list(temp['voltage'])}
    all_formation[name] = temp_dict

In [ ]:
all_results_capa = pd.DataFrame()
all_results_volt = {}

for i in list(rescaled_all_formation.keys()):
    X = []
    for j in range(120):
        X.append(np.array([rescaled_all_formation[i][0][j], rescaled_all_formation[i][1][j]]))
    X = np.array([X])
    
    result = model.predict(X, verbose=False)
    
    all_results_volt[i] = result[0][0]
    for c, k in enumerate(['1cyc', '50cyc']):
        all_results_capa.loc[i, k] = result[1][k][0][0]
    all_results_capa.loc[i, 'retention'] = result[1][1][0][0]/result[1][0][0][0]*100
    all_results_capa.loc[i, 'energy'] = sum(list(result[0][0])+[1.5])/21*result[1][1][0][0]
    
all_results_capa